# Integer Atlas — explore

Number-theory views over the dataset the CLI populated (read-only, via DuckDB).
Best with a sizable range, e.g. `integer-atlas fetch --start 1 --end 1000000 --columns "$(atlas-algos columns --csv)"`.
Each section notes the columns it needs.

In [ ]:
import math
import atlas
import plotly.express as px

con = atlas.connect()
def q(sql):
    return con.execute(sql).df()

lo, hi = con.execute('SELECT min(n), max(n) FROM numbers').fetchone()
print(f'loaded n = {lo} .. {hi}')
q('SELECT * FROM numbers ORDER BY n LIMIT 5')

## Primes — density and counting  
_needs `is_prime`._ Density of primes per interval decays like 1/ln n; the running count is π(x).

In [ ]:
d = q('SELECT (n // 10000) * 10000 AS bucket, avg(is_prime::int) AS density '
      'FROM numbers WHERE n > 1 GROUP BY 1 ORDER BY 1')
px.line(d, x='bucket', y='density', title='Prime density by interval (about 1/ln n)')

In [ ]:
pi = q('SELECT n, sum(is_prime::int) OVER (ORDER BY n) AS pi '
       'FROM numbers QUALIFY n % 2000 = 0')
px.line(pi, x='n', y='pi', title='Prime-counting function pi(x)')

## Ω(n) — the Erdős–Kac bell  
_needs `omega_big`._ The number of prime factors (with multiplicity) is bell-shaped; its mean grows like ln ln n.

In [ ]:
d = q('SELECT omega_big, count(*) AS c FROM numbers GROUP BY 1 ORDER BY 1')
px.bar(d, x='omega_big', y='c', title='Distribution of Omega(n)')

In [ ]:
d = q('SELECT n, omega_big FROM numbers LIMIT 1000')
px.line(d, x='n', y='omega_big', title='Plot of Omega(n)')

## Euler totient — the rays  
_needs `euler_phi`._ φ(n) plotted against n fans into striking rays (φ(n)/n clusters by prime structure).

In [ ]:
d = q('SELECT n, euler_phi FROM numbers WHERE n <= 3000')
fig = px.scatter(d, x='n', y='euler_phi', render_mode='webgl', title='phi(n) vs n — totient rays')
fig.update_traces(marker=dict(size=2))
fig

## Abundance and perfect numbers  
_needs `abundance_class`, `is_perfect`, `divisor_sum`._ Most numbers are deficient; perfect numbers are rare (6, 28, 496, 8128, …).

In [ ]:
cls = q('SELECT abundance_class, count(*) AS c FROM numbers GROUP BY 1 ORDER BY 2 DESC')
px.bar(cls, x='abundance_class', y='c', title='deficient / perfect / abundant')

In [ ]:
q('SELECT n, divisor_sum FROM numbers WHERE is_perfect ORDER BY n')

## Mertens function  
_needs `mobius`._ The running sum M(x) = Σ μ(n) wanders slowly — its growth is tied to the Riemann hypothesis.

In [ ]:
m = q('SELECT n, sum(mobius) OVER (ORDER BY n) AS M FROM numbers WHERE n < 10000 QUALIFY n % 10 = 0')
px.line(m, x='n', y='M', title='Mertens function M(x) = sum of mobius(n)')

## Collatz total stopping time  
_needs `collatz_stopping_time`._ Steps for the 3x+1 map to reach 1 — famously erratic.

In [ ]:
d = q('SELECT n, collatz_stopping_time AS steps FROM numbers WHERE n <= 10000')
fig = px.scatter(d, x='n', y='steps', render_mode='webgl', title='Collatz total stopping time')
fig.update_traces(marker=dict(size=2))
fig

## Squarefree density → 6/π²  
_needs `is_squarefree`._ The fraction of squarefree numbers converges to 6/π² ≈ 0.6079.

In [ ]:
d = q('SELECT n, avg(is_squarefree::int) OVER (ORDER BY n) AS dens FROM numbers QUALIFY n % 1000 = 0')
fig = px.line(d, x='n', y='dens', title='Squarefree density approaching 6/pi^2')
fig.add_hline(y=6 / math.pi**2, line_dash='dash')
fig

## 3D — popcount vs Ω(n) vs frequency  
_needs `binary_popcount`, `omega_big`._ How binary weight and prime-factor count co-occur.

In [ ]:
d3 = q('SELECT binary_popcount, omega_big, count(*) AS c FROM numbers GROUP BY 1, 2')
px.scatter_3d(d3, x='binary_popcount', y='omega_big', z='c', color='c', title='popcount vs Omega(n)')

## More to try

- **Sum of two squares** (`sum_of_two_squares_count`): fraction of n with `r2 > 0` per interval — Gauss-circle flavor.
- **Smooth numbers** (`largest_prime_factor`): histogram of `ln(largest_prime_factor)/ln(n)` — the Dickman curve.
- **Pólya's conjecture** (`liouville_lambda`): running sum of λ(n) — stays negative for a long time, then flips.
- **Highly composite numbers** (`divisor_count`): `argmax` of d(n) as n grows.
- **Digit patterns** (`digit_sum`, `digital_root`): Benford-like or periodic structure.

Publish any view as a dashboard with Voilà, or share notebooks with your team via JupyterHub.